# Frankie NSFW Generator — ComfyUI + Pony XL + IPAdapter FaceID

ComfyUI-based (handles model compat better than raw diffusers).

Runtime → Change runtime type → **T4 GPU** → Save

Runtime → **Run all** → walk ~12 min → 8 uncensored Frankie selfies appear.

In [ ]:
!nvidia-smi | head -15

In [ ]:
# ── 1. Install ComfyUI + IPAdapter Plus custom node ──
import os
if not os.path.exists('/content/ComfyUI'):
    !git clone -q https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt
if not os.path.exists('/content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus'):
    !git clone -q https://github.com/cubiq/ComfyUI_IPAdapter_plus /content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus
!pip install -q insightface onnxruntime-gpu opencv-python-headless
print('ComfyUI ready')

In [ ]:
# ── 2. Download all model weights ──
!mkdir -p /content/ComfyUI/models/checkpoints /content/ComfyUI/models/ipadapter /content/ComfyUI/models/loras /content/ComfyUI/models/clip_vision /content/ComfyUI/models/insightface/models/buffalo_l
# Pony Diffusion v6 XL
if not os.path.exists('/content/ComfyUI/models/checkpoints/pony_v6.safetensors'):
    !wget -q --show-progress -O /content/ComfyUI/models/checkpoints/pony_v6.safetensors https://huggingface.co/AstraliteHeart/pony-diffusion-v6/resolve/main/v6.safetensors
# IPAdapter FaceID Plus v2 SDXL (.bin + companion LoRA)
if not os.path.exists('/content/ComfyUI/models/ipadapter/ip-adapter-faceid-plusv2_sdxl.bin'):
    !wget -q --show-progress -O /content/ComfyUI/models/ipadapter/ip-adapter-faceid-plusv2_sdxl.bin https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl.bin
    !wget -q --show-progress -O /content/ComfyUI/models/loras/ip-adapter-faceid-plusv2_sdxl_lora.safetensors https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl_lora.safetensors
# CLIP vision encoder for IPAdapter
if not os.path.exists('/content/ComfyUI/models/clip_vision/CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors'):
    !wget -q --show-progress -O /content/ComfyUI/models/clip_vision/CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors
# InsightFace buffalo_l (downloaded automatically by IPAdapter node, but pre-fetch to avoid first-run slowness)
if not os.path.exists('/content/ComfyUI/models/insightface/models/buffalo_l/det_10g.onnx'):
    !wget -q -O /tmp/buffalo_l.zip https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip
    !unzip -q -o /tmp/buffalo_l.zip -d /content/ComfyUI/models/insightface/models/buffalo_l
!ls -lh /content/ComfyUI/models/checkpoints /content/ComfyUI/models/ipadapter /content/ComfyUI/models/clip_vision

In [ ]:
# ── 3. Fetch Frankie reference and put it in ComfyUI/input/ ──
import requests
FRANKIE_REF_URL = 'https://raw.githubusercontent.com/veyrapay/frankie-assets/main/frankie-ref.png'
os.makedirs('/content/ComfyUI/input', exist_ok=True)
with open('/content/ComfyUI/input/frankie-ref.png', 'wb') as f:
    f.write(requests.get(FRANKIE_REF_URL).content)
print('Frankie reference loaded into ComfyUI/input/frankie-ref.png')

In [ ]:
# ── 4. Start ComfyUI server in background ──
import subprocess, time, requests as r
subprocess.Popen(['python', '/content/ComfyUI/main.py', '--listen', '127.0.0.1', '--port', '8188', '--disable-auto-launch'], cwd='/content/ComfyUI')
# Wait for server to be ready
for i in range(60):
    try:
        r.get('http://127.0.0.1:8188/system_stats', timeout=2)
        print(f'ComfyUI up after {i*2}s')
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('ComfyUI did not start in 120s')

In [ ]:
# ── 5. Build workflow + submit 8 scenes ──
import json, uuid, time, requests as r
from PIL import Image
from IPython.display import display, Markdown

POS_PREFIX = 'score_9, score_8_up, score_7_up, source_photo, photorealistic, sharp focus, natural skin texture, freckles, dark wavy hair, gold hoop earrings, sleeve tattoos, '
NEGATIVE = 'score_4, score_5, score_6, source_furry, source_anime, source_cartoon, source_pony, deformed, bad anatomy, extra fingers, watermark, text, signature, blurry, low quality, ugly, mutated, disfigured'
SCENES = [
    'topless mirror selfie in bathroom, soft warm lighting, intimate',
    'topless in bed in morning light, smiling, candid',
    'wearing black lace lingerie, sitting on bed, looking at camera, sultry',
    'in a white t-shirt and panties, kitchen morning, holding coffee',
    'topless on a beach at sunset, ocean behind, hair wet',
    'in a sheer black robe, balcony at night, city lights behind',
    'taking a selfie in a black string bikini, pool behind, sun-kissed',
    'in a tight white tank top with no bra, gym mirror selfie, sweaty',
]

def build_workflow(prompt, seed):
    """Pony XL + IPAdapter FaceID Plus v2 + Frankie ref → 1024x1024 image."""
    return {
        '1': {'class_type': 'CheckpointLoaderSimple', 'inputs': {'ckpt_name': 'pony_v6.safetensors'}},
        '2': {'class_type': 'LoadImage', 'inputs': {'image': 'frankie-ref.png'}},
        '3': {'class_type': 'IPAdapterUnifiedLoaderFaceID', 'inputs': {'model': ['1', 0], 'preset': 'FACEID PLUS V2', 'lora_strength': 0.6, 'provider': 'CUDA'}},
        '4': {'class_type': 'IPAdapterFaceID', 'inputs': {'model': ['3', 0], 'ipadapter': ['3', 1], 'image': ['2', 0], 'weight': 0.8, 'weight_faceidv2': 1.0, 'weight_type': 'linear', 'combine_embeds': 'concat', 'start_at': 0.0, 'end_at': 1.0, 'embeds_scaling': 'V only'}},
        '5': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['1', 1]}},
        '6': {'class_type': 'CLIPTextEncode', 'inputs': {'text': NEGATIVE, 'clip': ['1', 1]}},
        '7': {'class_type': 'EmptyLatentImage', 'inputs': {'width': 1024, 'height': 1024, 'batch_size': 1}},
        '8': {'class_type': 'KSampler', 'inputs': {'model': ['4', 0], 'positive': ['5', 0], 'negative': ['6', 0], 'latent_image': ['7', 0], 'seed': seed, 'steps': 30, 'cfg': 7.0, 'sampler_name': 'dpmpp_2m', 'scheduler': 'karras', 'denoise': 1.0}},
        '9': {'class_type': 'VAEDecode', 'inputs': {'samples': ['8', 0], 'vae': ['1', 2]}},
        '10': {'class_type': 'SaveImage', 'inputs': {'images': ['9', 0], 'filename_prefix': 'frankie'}},
    }

client_id = str(uuid.uuid4())
results = []
for i, scene in enumerate(SCENES, 1):
    prompt = POS_PREFIX + scene
    wf = build_workflow(prompt, 42 + i)
    print(f'\n[{i}/{len(SCENES)}] {scene}')
    sub = r.post('http://127.0.0.1:8188/prompt', json={'prompt': wf, 'client_id': client_id}).json()
    if 'error' in sub:
        print('  submit error:', json.dumps(sub.get('error'))[:300], json.dumps(sub.get('node_errors',{}))[:500])
        continue
    pid = sub['prompt_id']
    # Poll history
    img_path = None
    for _ in range(120):
        time.sleep(2)
        h = r.get(f'http://127.0.0.1:8188/history/{pid}').json()
        if pid in h and h[pid].get('outputs'):
            outs = h[pid]['outputs'].get('10', {}).get('images', [])
            if outs:
                f = outs[0]
                img_url = f'http://127.0.0.1:8188/view?filename={f["filename"]}&subfolder={f["subfolder"]}&type={f["type"]}'
                img_path = f'/content/out_{i:02d}.png'
                with open(img_path, 'wb') as fh:
                    fh.write(r.get(img_url).content)
                print(f'  saved {img_path}')
                break
    if img_path:
        results.append((scene, img_path))
    else:
        print('  timed out')

print(f'\nDone. {len(results)}/{len(SCENES)} succeeded.')

In [ ]:
# ── 6. Display all results ──
from IPython.display import display, Markdown
from PIL import Image
for scene, path in results:
    display(Markdown(f'**{scene}**'))
    display(Image.open(path).resize((512, 512)))

In [ ]:
# ── 7. Zip & download ──
!cd /content && zip -q frankie_nsfw_batch.zip out_*.png
from google.colab import files
files.download('/content/frankie_nsfw_batch.zip')